In [1]:
# --------------------------------
# Imports & Path Setup
# --------------------------------

%reload_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path("..").resolve()))

import torch
import gc
import os
import pandas as pd

from src.models.restormer import RestormerModel
from src.models.autoencoder import ConvAutoencoder

from src.utils.trainer import Trainer
from src.utils.evaluation import ModelEvaluator

gc.collect()
torch.cuda.empty_cache()

In [2]:
# --------------------------------
# Paths
# --------------------------------

LOG_DIR = Path("logs")

RESIZED_DIR = Path("data/resized")
COMPRESSED_DIR = Path("data/compressed")
CHECKPOINTS_DIR = Path("checkpoints")

In [3]:
# --------------------------------
# Autoencoder
# --------------------------------

autoencoder = ConvAutoencoder()

trainer_autoencoder = Trainer(
    input_dir=RESIZED_DIR,
    compressed_dir=COMPRESSED_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    checkpoint_name="best_model_autoencoder",
    model=autoencoder,
    batch_size=4,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

trainer_autoencoder.load_dataset()

# --------------------------------
# Restormer
# --------------------------------

restormer = RestormerModel()

trainer_restormer = Trainer(
    input_dir=RESIZED_DIR,
    compressed_dir=COMPRESSED_DIR,
    checkpoints_dir=CHECKPOINTS_DIR,
    checkpoint_name="restormer",
    model=restormer,
    batch_size=4,
    learning_rate=1e-3,
    weight_decay=1e-4,
)

trainer_restormer.load_dataset()

Device : cuda
Parameters : 2,389,859
Train	| Val	| Test : 
700	| 150	| 150
Device : cuda
Parameters : 26,126,644
Train	| Val	| Test : 
700	| 150	| 150


In [4]:
# --------------------------------
# Experiments — (trainer, checkpoint_name)
# --------------------------------

experiments = {
    "autoencoder_both":     (trainer_autoencoder, "best_model_autoencoder_20260310_111257.pth"),
    "autoencoder_fourier":  (trainer_autoencoder, "best_model_autoencoder_fourier_only_20260327_102957.pth"),
    "autoencoder_wavelet":  (trainer_autoencoder, "best_model_autoencoder_wavelet_only_20260327_105234.pth"),
    "restormer_both":       (trainer_restormer,   "best_model_restormer_20260328_080420.pth"),
    "restormer_fourier":    (trainer_restormer,   "best_model_restormer_fourier_20260328_103041.pth"),
    "restormer_wavelet":    (trainer_restormer,   "best_model_restormer_wavelet_20260328_110824.pth"),
}

In [5]:
# --------------------------------
# Evaluation loop
# --------------------------------

results = {}

output_path = Path("results/evaluation_summary_complete.csv")
os.makedirs(output_path.parent, exist_ok=True)

for experiment_name, (trainer, checkpoint_name) in experiments.items():
    print(f"\n{'━' * 54}")
    print(f"  Evaluating: {experiment_name}")

    evaluator = ModelEvaluator(
        model=trainer.model,
        test_loader=trainer.test_loader,
        device=trainer.device,
        checkpoint=CHECKPOINTS_DIR / checkpoint_name,
        lpips_net=None,
    )

    results[experiment_name] = evaluator.evaluate(verbose=True)

    df = pd.DataFrame([results[experiment_name]])
    df.insert(0, "experiment", experiment_name)

    if output_path.exists():
        df.to_csv(output_path, mode="a", header=False, index=False)
    else:
        df.to_csv(output_path, mode="w", header=True, index=False)

    del evaluator
    gc.collect()
    torch.cuda.empty_cache()


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Evaluating: autoencoder_both
✔ Checkpoint loaded from checkpoints/best_model_autoencoder_20260310_111257.pth

──────────────────────────────────────────────────────
  Metric               Baseline   Restored      Delta
──────────────────────────────────────────────────────
  PSNR (dB)               25.60      26.62      +1.02
  SSIM                   0.7883     0.7849    -0.0034
  MSE                   0.00312    0.00278   -0.00035
  MAE                   0.04029    0.03690   -0.00339
──────────────────────────────────────────────────────


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Evaluating: autoencoder_fourier
✔ Checkpoint loaded from checkpoints/best_model_autoencoder_fourier_only_20260327_102957.pth

──────────────────────────────────────────────────────
  Metric               Baseline   Restored      Delta
──────────────────────────────────────────────────────
  PSNR (dB)               25.60      24.22      

In [7]:
import plotly.graph_objects as go
import pandas as pd

df = pd.read_csv("results/evaluation_summary_complete.csv")
labels = df["experiment"].str.replace("_", " ").str.title().tolist()

# Delta mae
colors = ["#2ecc71" if v >= 0 else "#e74c3c" for v in df["delta_mae"]]
fig = go.Figure(go.Bar(
    x=labels, y=df["delta_mae"],
    marker_color=colors,
    text=[f"{v:+.2f}" for v in df["delta_mae"]],
    textposition="outside",
))
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.update_xaxes(tickangle=-30)
fig.update_yaxes(title_text="Δmae (dB)")
fig.write_html("results/delta_mae.html")